# E5 · Persona VAE

Trains the VAE + GMM persona model from session feature vectors and compares it with
k-means on the raw features (silhouette). Runs on a laptop CPU in a minute; Colab works too.

Inputs: `data/persona/public_features.csv` from `python eval/extract_features.py --clips <folder of ~200 public speech clips>`
plus your own analysed sessions (`data/sessions/*/report.json`).

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "analytics").exists() else Path.cwd().parent
sys.path[:0] = [str(ROOT / "backend"), str(ROOT)]
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score
from analytics.persona.train import read_csv, session_vectors, train, fit_norms
from analytics.persona.features import to_array
from analytics.persona.infer import kmeans_raw
from pipeline.settings import get_settings

settings = get_settings()
csv_path = ROOT / "data" / "persona" / "public_features.csv"
vectors = (read_csv(csv_path) if csv_path.exists() else []) + session_vectors(settings.sessions_dir)
print(len(vectors), "vectors")

In [ ]:
result = train(vectors, settings.models_dir / "persona", latent=2, k=None, epochs=600, seed=settings.seed)
z, labels, x = result["z"], result["labels"], result["x"]
for p in result["meta"]["personas"]:
    print(f"{p['name']:<24} n={p['size']:<4} {p['description']}")

In [ ]:
k = len(result["meta"]["personas"])
km = kmeans_raw(vectors, k)
x_norm = np.stack([to_array(v, fit_norms(vectors)) for v in vectors])
scores = {
    "vae_gmm_silhouette_latent": silhouette_score(z, labels),
    "vae_gmm_silhouette_raw_space": silhouette_score(x_norm, labels),
    "kmeans_raw_silhouette_raw_space": silhouette_score(x_norm, km),
}
print(scores)
import csv
(ROOT / "eval" / "results").mkdir(parents=True, exist_ok=True)
with open(ROOT / "eval" / "results" / "e5_persona.csv", "w", newline="") as fh:
    w = csv.writer(fh); w.writerow(["metric", "value"]); [w.writerow([k_, v]) for k_, v in scores.items()]

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
names = {p["component"]: p["name"] for p in result["meta"]["personas"]}
for comp in sorted(set(labels)):
    m = labels == comp
    ax.scatter(z[m, 0], z[m, 1], s=14, label=names[comp])
ax.set_xlabel("latent 1"); ax.set_ylabel("latent 2"); ax.legend(fontsize=8); ax.set_title("Persona map (VAE latent, GMM clusters)")
fig.savefig(ROOT / "eval" / "results" / "e5_persona_map.png", dpi=150)